# ET Agent — Evolving Treatment Analysis

In [1]:
import pathlib
import sys
import os

import anthropic
import pandas as pd
from IPython.display import display, Markdown

_here = pathlib.Path().resolve()
if (_here / 'clinical-trial-optimizer').exists():
    CT_ROOT = _here / 'clinical-trial-optimizer'
elif (_here / 'src').exists():
    CT_ROOT = _here
elif (_here.parent / 'src').exists():
    CT_ROOT = _here.parent
else:
    raise RuntimeError(f'Cannot locate clinical-trial-optimizer root from {_here}')

sys.path.insert(0, str(CT_ROOT / 'src'))

from agent_prompt_functions import get_et_agent_prompt
from data_utils import filterETLOT, formatETLotDF

print(f'CT_ROOT : {CT_ROOT}')

CT_ROOT : /Users/rajeevkulkarni/ml-explorations/clinical-trial-optimizer


In [2]:
print('API key set' if os.environ.get('ANTHROPIC_API_KEY') else 'WARNING: ANTHROPIC_API_KEY not set')

API key set


In [3]:
DATA_FILE = CT_ROOT / 'data' / 'raw' /'rwd_lot.xlsx'

sheets = pd.read_excel(DATA_FILE, sheet_name=None)
print(f'Sheets found: {list(sheets.keys())}')

# Adjust sheet names to match your workbook
period1_raw = sheets['lot_c1']
period2_raw = sheets['lot_c2']

PERIOD1_RANGE = '2017-2020'
PERIOD2_RANGE = '2021-2024'

et_raw = filterETLOT(period1_raw, period2_raw)
et_df = formatETLotDF(et_raw)

print(f'Rows after filterETLOT: {len(et_raw)}')
print(et_df[:500])

Sheets found: ['soc_lot', 'lot_c1', 'lot_c2']
Rows after filterETLOT: 33
| Regimen                              | Line Number   | Period 1 % (Per Line)   | Period 2 % (Per Line)   | delta   | Gap Type   | Trend      |
|:-------------------------------------|:--------------|:------------------------|:------------------------|:--------|:-----------|:-----------|
| gefitinib                            | L1R3          | nan                     | 8.0%                    | 8.0%    | Emerging   | Emerging   |
| amivantamab+carboplatin+pemetrexed   | L2R5          | nan     


In [4]:
# Paste I/E criteria here
ie_criteria = """
Inclusion Criteria:
- Participant must have at least 1 measurable lesion, according to Response Evaluation Criteria in Solid Tumors (RECIST) version 1.1, that has not been previously irradiated
- Participant must have histologically or cytologically confirmed, locally advanced or metastatic, non-squamous non-small cell lung cancer (NSCLC), characterized at or after the time of locally advanced or metastatic disease diagnosis by either epidermal growth factor receptor (EGFR) Exon 19del or Exon 21 L858R mutation
- A participant with a history of brain metastases must have had all lesions treated as clinically indicated (that is, no current indication for further definitive local therapy). Any definitive local therapy to brain metastases must have been completed at least 14 days prior to randomization and the participant can be receiving no greater than10 milligrams (mg) prednisone or equivalent daily for the treatment of intracranial disease
- Participant must have Eastern Cooperative Oncology Group (ECOG) status of 0 or 1
- Any toxicities from prior systemic anticancer therapy must have resolved to National Cancer Institute Common Terminology Criteria for Adverse Events (NCI CTCAE) Version 5.0 Grade 1 or baseline level (except for alopecia [any grade], Grade <= 2 peripheral neuropathy, or Grade <= 2 hypothyroidism stable on hormone replacement)
- A participant of childbearing potential must have a negative serum pregnancy test at screening and within 72 hours of the first dose of study treatment and must agree to further serum or urine pregnancy tests during the study
- Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second- line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy. Participants who received either neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose of such therapy and then the participant progressed on or after osimertinib in the locally advanced or metastatic setting. Treatment with osimertinib must be discontinued at least 8 days (4 half-lives) prior to randomization (that is last dose no later than Day -8)

Exclusion Criteria:
- Participant received radiotherapy for palliative treatment of NSCLC less than 14 days prior to randomization
- Participant with symptomatic or progressive brain metastases
- Participant has history of or current evidence of leptomeningeal disease, or participant has spinal cord compression not definitively treated with surgery or radiation
- Participant has known small cell transformation
- Participant has a medical history of interstitial lung disease (ILD), including drug-induced ILD or radiation pneumonitis
- Participant has a history of clinically significant cardiovascular disease including, but not limited to diagnosis of deep vein thrombosis or pulmonary embolism within 4 weeks prior to randomization; myocardial infarction; unstable angina; stroke; transient ischemic attack; coronary/peripheral artery bypass graft; or acute coronary syndrome. Participant has a significant genetic predisposition to venous thromboembolic events. Participant has a prior history of venous thromboembolic events and is not on appropriate therapeutic anticoagulation as per National Comprehensive Cancer Network or local guidelines"
"""

print(f'Length of I/E criteria: {len(ie_criteria)} characters')

Length of I/E criteria: 3585 characters


In [5]:
et_prompt = get_et_agent_prompt(ie_criteria, et_df, period1_range=PERIOD1_RANGE, period2_range=PERIOD2_RANGE)
print(f'Prompt length: {len(et_prompt)} chars')

Prompt length: 27870 chars


In [8]:
# ⚠️ HUMAN APPROVAL REQUIRED — this cell calls the Claude API
client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=8096,
    system="You are a biomedical trial analysis agent. Output ONLY the final markdown sections as instructed. Do not output internal reasoning, step outputs, or any text before the STEP 1 verification line. Do not truncate output.",
    messages=[{"role": "user", "content": et_prompt}]
)

display(Markdown(response.content[0].text))

## STEP 1 Verification

- Total Emerging regimens identified: 9
- Total Major regimens identified: 10
- Total Minor+Decreasing regimens identified: 0
- Grand total: 19

---

## Gap Analysis Table

| Type | Original Criterion | Gap Type | Queriable | Patient Share (Period 1) | Patient Share (Period 2) | Temporal Trend | Explanation | Recommendation |
|------|--------------------|----------|-----------|--------------------------|--------------------------|----------------|-------------|----------------|
| Inclusion | Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second-line setting after prior treatment with first- or second-generation EGFR TKI as a monotherapy. | None | Yes | **osimertinib L1:** 5.0% (L1R6); **osimertinib L2:** 64.0% (L2R1) | **osimertinib L1:** 61.4% (L1R1); **osimertinib L2:** 30.0% (L2R1) | ↑ | The trial requires prior **osimertinib** in either first- or second-line. **Period 1 (2017–2020):** **osimertinib** was predominantly a second-line therapy (**64.0% L2R1**) with minimal first-line use (**5.0% L1R6**). **Period 2 (2021–2024):** **osimertinib** has dramatically shifted to dominant first-line use (**61.4% L1R1**) with reduced second-line use (**30.0% L2R1**). This criterion is well-aligned with the evolving real-world landscape. Already captured by existing I/E Criteria. | Already captured by existing I/E Criteria |
| Inclusion | Osimertinib must have been administered as either the first-line treatment or in the second-line setting after prior treatment with first- or second-generation EGFR TKI as a monotherapy. | None | Yes | **gefitinib L1:** 28.0% (L1R1); **erlotinib L1:** 24.0% (L1R2); **afatinib L1:** 19.0% (L1R3); **dacomitinib L1:** 10.0% (L1R4) | **gefitinib L1:** 8.0% (L1R3); **erlotinib L1:** 6.0% (L1R4); **afatinib L1:** 5.0% (L1R5); **dacomitinib L1:** 3.0% (L1R6) | ↓ | The criterion explicitly permits prior first- or second-generation EGFR TKI (gefitinib, erlotinib, afatinib, dacomitinib) in L1 preceding osimertinib in L2. **Period 1 (2017–2020):** first/second-generation TKIs dominated L1: **gefitinib 28.0%**, **erlotinib 24.0%**, **afatinib 19.0%**, **dacomitinib 10.0%**. **Period 2 (2021–2024):** all are substantially declining (**gefitinib 8.0%**, **erlotinib 6.0%**, **afatinib 5.0%**, **dacomitinib 3.0%**) as osimertinib has captured first-line dominance. Already captured by existing I/E Criteria. | Already captured by existing I/E Criteria |
| Suggested Addition: Inclusion | New Criterion | Major | Yes | 16.0% (L2R2) | 25.07% (L2R2) | ↑ | **carboplatin+pemetrexed** is a Major, Increasing regimen in L2 (**Period 1 (2017–2020): 16.0% L2R2**; **Period 2 (2021–2024): 25.07% L2R2**). The current I/E requires only prior osimertinib monotherapy. Patients who received **carboplatin+pemetrexed** as second-line therapy after a first/second-generation TKI (before subsequently receiving osimertinib in later lines) could represent an eligible population not explicitly addressed. However, as the trial targets post-osimertinib progression and osimertinib must be the most recent line, this regimen in L2 could reflect prior platinum-based therapy before osimertinib in L1. This is a notable real-world pattern warranting explicit inclusion consideration. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+pemetrexed as second-line systemic therapy prior to osimertinib administered as a subsequent line. |
| Suggested Addition: Inclusion | New Criterion | Major | Yes | N/A (not in Period 1 dataset) | 15.07% (L2R3) | ⚠ | **carboplatin+pemetrexed+bevacizumab** is a Major, Increasing regimen in L2 (**Period 1 (2017–2020): not present**; **Period 2 (2021–2024): 15.07% L2R3**). This is an emerging real-world second-line regimen that patients may have received prior to osimertinib. The current I/E does not explicitly address participants who received platinum-based bevacizumab-containing regimens as prior therapy. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+pemetrexed+bevacizumab as second-line systemic therapy prior to osimertinib administered as a subsequent line. |
| Suggested Addition: Inclusion | New Criterion | Major | Yes | N/A (not in Period 1 dataset) | 11.07% (L2R4) | ⚠ | **carboplatin+pemetrexed+pembrolizumab** is a Major, Increasing regimen in L2 (**Period 1 (2017–2020): not present**; **Period 2 (2021–2024): 11.07% L2R4**). This chemo-immunotherapy combination is being used in second-line in real-world practice. Patients who received this regimen before osimertinib are not explicitly addressed in current I/E. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+pemetrexed+pembrolizumab as second-line systemic therapy prior to osimertinib administered as a subsequent line. |
| Suggested Addition: Inclusion | New Criterion | Major | Yes | N/A (not in Period 1 dataset) | 7.07% (L2R5) | ⚠ | **amivantamab+carboplatin+pemetrexed** is a Major, Increasing regimen in L2 (**Period 1 (2017–2020): not present**; **Period 2 (2021–2024): 7.07% L2R5**). This is an emerging standard-of-care combination increasingly used in second-line EGFR-mutant NSCLC following osimertinib. As a post-osimertinib regimen in L2, this does not represent a prior-to-osimertinib therapy context; rather, it represents competition with the trial's own treatment. This criterion warrants explicit acknowledgment. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have progressed on or after amivantamab+carboplatin+pemetrexed administered as second-line systemic therapy. |
| Suggested Addition: Inclusion | New Criterion | Emerging | Yes | N/A (not in Period 1 dataset) | 5.07% (L2R6) | ⚠ | **docetaxel** is an Emerging regimen in L2 (**Period 1 (2017–2020): not present**; **Period 2 (2021–2024): 5.07% L2R6**). Real-world second-line use of docetaxel is growing. Patients who received docetaxel in L2 prior to a subsequent osimertinib course are not explicitly addressed in the I/E criteria. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received docetaxel as second-line systemic therapy prior to osimertinib administered as a subsequent line. |
| Suggested Addition: Inclusion | New Criterion | Emerging | Yes | N/A (not in Period 1 dataset) | 3.07% (L2R7) | ⚠ | **carboplatin+paclitaxel** is an Emerging regimen in L2 (**Period 1 (2017–2020): not present**; **Period 2 (2021–2024): 3.07% L2R7**). Real-world second-line use of carboplatin+paclitaxel is emerging. Patients who received this regimen before osimertinib in a later line are not explicitly addressed in current I/E. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+paclitaxel as second-line systemic therapy prior to osimertinib administered as a subsequent line. |
| Suggested Addition: Inclusion | New Criterion | Major | Yes | N/A (not in Period 1 dataset) | 12.0% (L1R2) | ⚠ | **osimertinib+carboplatin+pemetrexed** is a Major, Increasing regimen in L1 (**Period 1 (2017–2020): not present**; **Period 2 (2021–2024): 12.0% L1R2**). The current I/E specifies osimertinib *monotherapy* as the qualifying prior treatment. Patients who received **osimertinib in combination** with carboplatin+pemetrexed as first-line therapy are not explicitly included and represent a substantial and growing real-world population. | Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have progressed on or after osimertinib+carboplatin+pemetrexed administered as first-line systemic therapy. |

---

## Updated Treatment-Related Criteria

**Based on Original Criterion:** *Participant must have progressed on or after osimertinib monotherapy as the most recent line of treatment. Osimertinib must have been administered as either the first-line treatment for locally advanced or metastatic disease or in the second-line setting after prior treatment with first- or second-generation EGFR tyrosine kinase inhibitor (TKI) as a monotherapy.*

- **Suggested Addition 1 (Gap Type: Major ↑):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have progressed on or after osimertinib+carboplatin+pemetrexed administered as first-line systemic therapy.
- **Suggested Addition 2 (Gap Type: Major ↑):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+pemetrexed as second-line systemic therapy prior to osimertinib administered as a subsequent line.
- **Suggested Addition 3 (Gap Type: Major ⚠):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+pemetrexed+bevacizumab as second-line systemic therapy prior to osimertinib administered as a subsequent line.
- **Suggested Addition 4 (Gap Type: Major ⚠):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+pemetrexed+pembrolizumab as second-line systemic therapy prior to osimertinib administered as a subsequent line.
- **Suggested Addition 5 (Gap Type: Major ⚠):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have progressed on or after amivantamab+carboplatin+pemetrexed administered as second-line systemic therapy.
- **Suggested Addition 6 (Gap Type: Emerging ⚠):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received docetaxel as second-line systemic therapy prior to osimertinib administered as a subsequent line.
- **Suggested Addition 7 (Gap Type: Emerging ⚠):** Include participants with locally advanced or metastatic non-squamous NSCLC harboring EGFR exon 19 deletion or L858R mutation who have received carboplatin+paclitaxel as second-line systemic therapy prior to osimertinib administered as a subsequent line.

**Updated Combined Criterion:** Participant must have progressed on or after osimertinib (as monotherapy or in combination with carboplatin+pemetrexed) as the most recent line of treatment. Osimertinib (or osimertinib+carboplatin+pemetrexed) must have been administered as either the first-line treatment for locally advanced or metastatic disease, or in the second-line or later setting after prior treatment with first- or second-generation EGFR TKI as monotherapy, or after prior platinum-based chemotherapy regimens (including carboplatin+pemetrexed, carboplatin+pemetrexed+bevacizumab, carboplatin+pemetrexed+pembrolizumab, carboplatin+paclitaxel, or docetaxel) or amivantamab+carboplatin+pemetrexed administered in the second-line setting. Participants who received neoadjuvant and/or adjuvant treatment of any type are eligible if progression to locally advanced or metastatic disease occurred at least 12 months after the last dose of such therapy and then the participant progressed on or after osimertinib in the locally advanced or metastatic setting. Treatment with osimertinib must be discontinued at least 8 days (4 half-lives) prior to randomization.

---

## Therapeutic Context Summary

- **Disease & Biomarker Context:** Locally advanced or metastatic, non-squamous non-small cell lung cancer (NSCLC) with EGFR Exon 19 deletion or Exon 21 L858R mutation; post-osimertinib progression (third-line or later trial setting).

- **Temporal Treatment Landscape Evolution:**
  - **First-line:** The most dramatic shift is the rise of **osimertinib monotherapy** from minimal first-line use (**5.0%** in **Period 1 (2017–2020)**) to dominance (**61.4%** in **Period 2 (2021–2024)**), displacing legacy first/second-generation EGFR TKIs (gefitinib **28.0% → 8.0%**; erlotinib **24.0% → 6.0%**; afatinib **19.0% → 5.0%**; dacomitinib **10.0% → 3.0%**). A new combination **osimertinib+carboplatin+pemetrexed** emerged in **Period 2** at **12.0% (L1R2)**, representing a meaningful first-line alternative not present in **Period 1**.
  - **Second-line:** **osimertinib** as second-line decreased from **64.0% (Period 1)** to **30.0% (Period 2)**. New Major regimens emerged in **Period 2**: **carboplatin+pemetrexed** grew from **16.0% → 25.07%**; **carboplatin+pemetrexed+bevacizumab** emerged at **15.07%**; **carboplatin+pemetrexed+pembrolizumab** emerged at **11.07%**; **amivantamab+carboplatin+pemetrexed** emerged at **7.07%**.
  - These patterns confirm that the post-osimertinib treatment pathway is rapidly diversifying in **Period 2**, creating a more complex eligibility landscape.

- **Line of Therapy Alignment:** The trial targets patients post-osimertinib, placing them predominantly in third-line (L3) in **Period 2** given osimertinib's first-line dominance. In **Period 1**, post-osimertinib patients were more commonly in third-line after first-generation TKI → osimertinib sequences (L1 TKI → L2 osimertinib **64.0%**). In **Period 2**, the population is predominantly post-first-line osimertinib (**61.4% L1**), making the trial's target population well-aligned with the dominant second-line and beyond landscape. The emergence of **osimertinib+carboplatin+pemetrexed** (**12.0% L1R2**) as a first-line combination creates a new subgroup not addressed by the current "osimertinib monotherapy" eligibility language.

- **Key Observations:**
  - The trial's eligibility criterion specifying **osimertinib monotherapy** as the qualifying prior treatment is increasingly misaligned with real-world practice in **Period 2 (2021–2024)**, where **osimertinib+carboplatin+pemetrexed** accounts for **12.0%** of first-line use and is classified as a Major Increasing regimen.
  - The second-line landscape has diversified substantially in **Period 2**, with **carboplatin+pemetrexed (25.07%)**, **carboplatin+pemetrexed+bevacizumab (15.07%)**, and **amivantamab+carboplatin+pemetrexed (7.07%)** all representing common treatment pathways between first-generation TKI and eventual osimertinib use—or as post-osimertinib regimens—that are not explicitly addressed in the I/E criteria.
  - Legacy first/second-generation EGFR TKIs, while explicitly permitted as prior therapy in the I/E, are sharply declining in **Period 2**, reducing the pool of patients eligible via the L1 TKI → L2 osimertinib pathway.
  - **amivantamab+carboplatin+pemetrexed** appearing in both L2 (**7.07%**) and L3 (**15.0%**) in **Period 2** signals its rapid emergence as a post-osimertinib standard of care, directly relevant to this trial's competitive landscape.

- **Primary Recommendation:** The trial's I/E criteria are well-calibrated to **Period 1 (2017–2020)** practice patterns (second-line osimertinib after first-generation TKI), but are increasingly misaligned with **Period 2 (2021–2024)** reality, where first-line **osimertinib monotherapy (61.4%)** and combination **osimertinib+carboplatin+pemetrexed (12.0%)** dominate. The most impactful single update is to explicitly extend eligibility to patients who progressed on **osimertinib+carboplatin+pemetrexed** as first-line therapy, ensuring the trial captures the largest and fastest-growing segment of the EGFR-mutant locally advanced or metastatic non-squamous NSCLC population in the current treatment era.